# FRI-BCI: EA-EEGNet Pretraining, CRC-Hysteresis Calibration & Phase C Benchmark
**Hardware Target: Google Colab GPU (`cuda` / A100 / L4 / T4) or CPU fallback**

This notebook launches the repository's validated Stage 1 and Stage 2 pipeline (`python -m eeg_decoder.gpu_pipeline`) configured by `configs/gpu_research.json`:
- **Source pretraining (`pretrain`)**: 89 PhysioNet source subjects (`S016`–`S109` excluding known 128 Hz/anomalous IDs `88, 89, 92, 100, 104`), fitting Euclidean Alignment (EA) on `R04`, training `EEGNet` on `R04`/`R08`, and validating on `R12`.
- **Target adaptation (`adapt`)**: 15 target subjects (`S001`–`S015`: `S001`–`S010` previously inspected / development, `S011`–`S015` unopened prospective), fitting EA and adapting `EEGNet` (both `transfer` and `--scratch` controls) strictly on `R04`.
- **Target gate calibration (`calibrate`)**: Episode-level CRC-adjusted threshold selection for `HysteresisGate` strictly on `R08` (`guarantee_claimed: False`).
- **Target evaluation (`evaluate`)**: Opens held-out `R12` only when explicitly invoked.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal/blob/codex/robot-learning-design/notebooks/FRI_BCI_Colab_Master.ipynb)

In [ ]:
# [Step 1] Clone or update repository, install MNE without breaking Colab's NumPy/PyTorch ABI, and validate protocol
import importlib.util
import os
import pathlib
import subprocess
import sys

def run_cmd(cmd):
    print("$", " ".join(str(x) for x in cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

REPO_URL = "https://github.com/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal.git"
BRANCH = "codex/robot-learning-design"
if pathlib.Path("/content").exists():
    REPO_DIR = pathlib.Path("/content/repo")
elif (pathlib.Path.cwd() / "configs/gpu_research.json").exists():
    REPO_DIR = pathlib.Path.cwd()
elif (pathlib.Path.cwd().parent / "configs/gpu_research.json").exists():
    REPO_DIR = pathlib.Path.cwd().parent
    os.chdir(REPO_DIR)
else:
    REPO_DIR = pathlib.Path.cwd()

if REPO_DIR == pathlib.Path("/content/repo"):
    if not (REPO_DIR / ".git").exists():
        run_cmd(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH])
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
        run_cmd(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"])
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    missing = [pkg for pkg in ("mne", "pyriemann") if importlib.util.find_spec(pkg) is None]
    if missing:
        run_cmd([sys.executable, "-m", "pip", "install", "-q", *missing])

run_cmd([sys.executable, "-m", "eeg_decoder.gpu_pipeline", "validate", "--config", "configs/gpu_research.json"])


In [ ]:
# [Step 2] Stage 1: Source Pretraining (89 source subjects: R04/R08 train, R12 validation)
# Tip: For a fast 2-subject smoke test on Colab before running all 89 subjects, keep QUICK_PILOT = True.
# Set QUICK_PILOT = False to run the full 89-subject pretraining.
import json
import pathlib
import sys
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUICK_PILOT = False

if QUICK_PILOT:
    WORK_DIR = "outputs/gpu_research_pilot"
    base_cfg = json.loads(pathlib.Path("configs/gpu_research.json").read_text(encoding="utf-8"))
    base_cfg["source_subjects"] = [16, 17, 18, 19]
    base_cfg["pretrain_epochs"] = 10
    pathlib.Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
    CONFIG = f"{WORK_DIR}/pilot_config.json"
    pathlib.Path(CONFIG).write_text(json.dumps(base_cfg, indent=2), encoding="utf-8")
else:
    WORK_DIR = "outputs/gpu_research"
    CONFIG = "configs/gpu_research.json"

print(f"Running pretrain stage on device={DEVICE} (QUICK_PILOT={QUICK_PILOT})...")
run_cmd([
    sys.executable, "-m", "eeg_decoder.gpu_pipeline", "pretrain",
    "--config", CONFIG,
    "--work-dir", WORK_DIR,
    "--device", DEVICE,
    "--resume",
])


In [ ]:
# [Step 3] Stage 1 & 2: Target Adaptation (R04) and Episode-Level CRC Calibration (R08)
# By default, runs on exploratory/previously-inspected subjects (S001-S010).
# Keep RUN_EVALUATE_R12 = False until ready to open R12 for a target cohort.
import pathlib
import sys

DEVELOPMENT_TARGETS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
PROSPECTIVE_TARGETS = [11, 12, 13, 14, 15]
RUN_PROSPECTIVE_COHORT = False
RUN_EVALUATE_R12 = False

targets = DEVELOPMENT_TARGETS + (PROSPECTIVE_TARGETS if RUN_PROSPECTIVE_COHORT else [])

for subject in targets:
    for kind, extra_flags in (("transfer", []), ("scratch", ["--scratch"])):
        subj_dir = pathlib.Path(WORK_DIR) / kind / f"S{subject:03d}"
        run_cmd([
            sys.executable, "-m", "eeg_decoder.gpu_pipeline", "adapt",
            "--config", CONFIG, "--work-dir", WORK_DIR, "--device", DEVICE,
            "--subject", str(subject), "--resume", *extra_flags,
        ])
        if not (subj_dir / "calibration.json").exists():
            run_cmd([
                sys.executable, "-m", "eeg_decoder.gpu_pipeline", "calibrate",
                "--config", CONFIG, "--work-dir", WORK_DIR, "--device", DEVICE,
                "--subject", str(subject), *extra_flags,
            ])
        if RUN_EVALUATE_R12 and not (subj_dir / "evaluation").exists():
            run_cmd([
                sys.executable, "-m", "eeg_decoder.gpu_pipeline", "evaluate",
                "--config", CONFIG, "--work-dir", WORK_DIR, "--device", DEVICE,
                "--subject", str(subject), *extra_flags,
            ])


In [ ]:
# [Step 4] Inspect Calibrated Thresholds and (if evaluated) Held-Out R12 Metrics
import json
import pathlib

work = pathlib.Path(WORK_DIR)
print("=== R08 Calibrated Thresholds ===")
for kind in ("transfer", "scratch"):
    for cal_file in sorted((work / kind).glob("S*/calibration.json")):
        cal = json.loads(cal_file.read_text(encoding="utf-8"))
        print(
            f"{kind:8s} | S{cal['subject']:03d} | threshold={cal['threshold']:.3f} "
            f"| status={cal['status']} | crc_adj_risk={cal['selected_crc_adjusted_risk']}"
        )

eval_files = list(work.glob("*/S*/evaluation/metrics.json"))
if eval_files:
    print("\n=== R12 Evaluation Summary ===")
    for m_file in sorted(eval_files):
        m = json.loads(m_file.read_text(encoding="utf-8"))
        kind = m_file.parents[2].name
        w = m["window_metrics"]
        c_crc = m["command_metrics"]["crc_hysteresis"]
        c_raw = m["command_metrics"]["raw"]
        print(
            f"{kind:8s} | S{m['subject']:03d} | BA={w['balanced_accuracy']:.3f} "
            f"| Raw REST cmd={c_raw['rest_command_fraction']:.3f} "
            f"| CRC REST cmd={c_crc['rest_command_fraction']:.3f} "
            f"| CRC Missed Imagery={c_crc['missed_imagery_fraction']:.3f}"
        )
